## Transform customer Data
### 1. Remove records with null customer_id
### 2. remove exact duplicate records
### 3. remove duplicate records based on creatd_timestamp
### 4. Cast the columns to the correct data type
### 5. Write transformed data to the silver schema

In [0]:
select * from gizmobox_catalog_noori.bronze.v_customers
where customer_id is not null

In [0]:
select * from gizmobox_catalog_noori.bronze.v_customers
where customer_id is not null
order by customer_id



In [0]:
select distinct * 
from gizmobox_catalog_noori.bronze.v_customers
where customer_id is not null
order by customer_id


In [0]:
select customer_id,
max(created_timestamp),
max(customer_name),
max(date_of_birth),
max(email),
max(member_since),
max(telephone),
max(filepath)
from gizmobox_catalog_noori.bronze.v_customers
where customer_id is not null
group by customer_id 

order by customer_id

In [0]:
create or replace temporary view v_customers_distinct as
select distinct * 
from gizmobox_catalog_noori.bronze.v_customers
where customer_id is not null
order by customer_id

In [0]:
with cte_max as (
select 
customer_id,
max(created_timestamp) as max_created_timestamp
from v_customers_distinct
group by customer_id
)   
select t.* 
from  v_customers_distinct t
join cte_max m on m.customer_id=t.customer_id and m.max_created_timestamp=t.created_timestamp
order by customer_id
 


In [0]:


with cte_max as (
select 
customer_id,
max(created_timestamp) as max_created_timestamp
from v_customers_distinct
group by customer_id
)   
select 
cast(t.created_timestamp as timestamp) as created_timestamp,
t.customer_id,
t.customer_name,
cast(t.date_of_birth as date) as date_of_birth,
t.email,
cast(t.member_since as date) as member_since,
t.telephone
from  v_customers_distinct t
join cte_max m on m.customer_id=t.customer_id and m.max_created_timestamp=t.created_timestamp
order by customer_id

In [0]:
create table gizmobox_catalog_noori.silver.customers
as 
with cte_max as (
select 
customer_id,
max(created_timestamp) as max_created_timestamp
from v_customers_distinct
group by customer_id
)   
select 
cast(t.created_timestamp as timestamp) as created_timestamp,
t.customer_id,
t.customer_name,
cast(t.date_of_birth as date) as date_of_birth,
t.email,
cast(t.member_since as date) as member_since,
t.telephone
from  v_customers_distinct t
join cte_max m on m.customer_id=t.customer_id and m.max_created_timestamp=t.created_timestamp
order by customer_id

In [0]:
select * from gizmobox_catalog_noori.silver.customers

In [0]:
describe extended gizmobox_catalog_noori.silver.customers